In [ ]:
# CÉLULA DE TESTE DA BIBLIOTECA CYVCF2

from cyvcf2 import VCF

FILE_NAME = 'Y.vcf'
FILE_PATH = f'/home/marcela/IC/IC/files/{FILE_NAME}'

# Abre o arquivo .vcf (ou .vcf.gz)
vcf = VCF(FILE_PATH)

#Caso se queira visualizar somente o header
#print(vcf.raw_header)

#visualização das amostras participantes
print("amostras: ", vcf.samples)

#Iterando sobre as variantes(linhas do Vcf)
for variant in vcf:

  #Impressão da linha completa
  print("Variante: ", variant)
  
  #Leitura de dados básicos
  cromossomo = variant.CHROM
  posicao = variant.POS
  id = variant.ID
  ref = variant.REF
  alt = variant.ALT
  quality = variant.QUAL

  #Acessando o campo INFO
  dp = variant.INFO.get('DP')
  ac = variant.INFO.get('AC')
  af = variant.INFO.get('AF')

  #Acessando os genótipos das amostras
  genotipos = variant.genotypes
  gt_types = variant.gt_types
  gt_ref_depths = variant.gt_ref_depths
  gt_alt_depths = variant.gt_alt_depths
  gt_phases = variant.gt_phases
  gt_quals = variant.gt_quals
  gt_bases = variant.gt_bases

  print(f'CHROM: {cromossomo}\nPOS: {posicao}\nID: {id}\nREF: {ref}\nALT: {alt}\nQUAL: {quality:.2f}\n')
  print(genotipos)
  # print(gt_types)
  # print(gt_alt_depths)
  # print(gt_ref_depths)
  # print(gt_phases)
  # print(gt_quals)
  # print(gt_bases)

  #Forma de acessar informações especificas do campo FORMAT (para uma determinada amostra deve-se especificar o idx) 
  # (ex: os resultados de DP daquela amostra em todas as variantes)
  sample_idx = 1
  dp_array = variant.format('DP')
  dp = dp_array[sample_idx].tolist() if dp_array is not None else None

  #Para imprimir apenas o primeiro
  break

vcf.close()


In [ ]:
import pandas as pd
import os
from enum import Enum
from cyvcf2 import VCF


FILE_NAME = 'Y.vcf'
# FILE_NAME = 'Cyberseg_chr21.vcf'

FILE_PATH = f'/home/marcela/IC/IC/files/{FILE_NAME}'

VCF_FILE = VCF(FILE_PATH)

class BASIC_COLS(Enum):
    CHROM = 0
    POS = 1
    ID = 2
    REF = 3
    ALT = 4
    QUAL = 5
    FILTER = 6
    INFO = 7  
    FORMAT = 8


def vcf_to_df_raw(vcf_path, vcf_file):
    cmd = "zgrep '^#' " + vcf_path + "|tail -n 1"
    cols = os.popen(cmd).read().strip('#').strip('\n').split('\t')
    
    data = []
    
    for variant in vcf_file:
        raw_line = str(variant).rstrip('\n').split('\t')
        data.append(raw_line)

    df = pd.DataFrame(data, columns=cols)
    return df

def vcf_to_df_filtered_INFO(vcf_file):
    #Define as colunas desejadas e define subcolunas para pegar apenas parte de INFO
    cols_tuples = [
        ('#CHROM', ''),
        ('POS', ''),
        ('REF', ''),
        ('ALT', ''),
        ('INFO', 'AC'),  
        ('INFO', 'AF'),  
        ('INFO', 'DP'),  
        ('FORMAT', '')
    ]

    #Acrescenta as colunas de amostras
    for sample in vcf_file.samples:
        cols_tuples.append((sample, ''))

    multi_cols = pd.MultiIndex.from_tuples(cols_tuples)

    data = []
    for variant in vcf_file:
        raw_line = str(variant).strip('\n').split('\t')

        fltrd_line = [
        	raw_line[BASIC_COLS.CHROM.value],
            raw_line[BASIC_COLS.POS.value],
            raw_line[BASIC_COLS.REF.value],
            raw_line[BASIC_COLS.ALT.value],
            variant.INFO.get('AC'),
            variant.INFO.get('AF'),
            variant.INFO.get('DP'),
            raw_line[BASIC_COLS.FORMAT.value] 
        ]

        fltrd_line.extend(raw_line[BASIC_COLS.FORMAT.value+1:])
        data.append(fltrd_line)
    
    df = pd.DataFrame(data, columns=multi_cols)
    return df

def vcf_to_df_filtered_Samples(vcf_file):
    #Define as colunas desejadas e define subcolunas para pegar apenas parte de INFO
    cols_tuples = [
        ('#CHROM', ''),
        ('POS', ''),
        ('REF', ''),
        ('ALT', ''),
        ('INFO', 'AC'),  
        ('INFO', 'AF'),  
        ('INFO', 'DP'),  
    ]

    #Acrescenta as colunas de amostras
    for sample in vcf_file.samples:
        sample_tuple = [
            (sample, 'GT'),
            (sample, 'AF'),
            (sample, 'DP'),
        ]
        cols_tuples.extend(sample_tuple)

    multi_cols = pd.MultiIndex.from_tuples(cols_tuples)

    data = []
    for variant in vcf_file:
        raw_line = str(variant).strip('\n').split('\t')

        fltrd_line = [
        	raw_line[BASIC_COLS.CHROM.value],
            raw_line[BASIC_COLS.POS.value],
            raw_line[BASIC_COLS.REF.value],
            raw_line[BASIC_COLS.ALT.value],
            variant.INFO.get('AC'),
            variant.INFO.get('AF'),
            variant.INFO.get('DP'),
        ]

        # Extração com cyvcf2. Retorna arrays ou None.
        genotypes = variant.genotypes
        af_array = variant.format('AF')
        dp_array = variant.format('DP')

        samples_fltrd_line = []
        for i in range(len(vcf_file.samples)):
            # Pegamos o valor se o array existir, senão colocamos "."
            gt_val = genotypes[i] if genotypes is not None else "."
            af_val = af_array[i][0] if af_array is not None else "."
            dp_val = dp_array[i][0] if dp_array is not None else "."

            sample_fltrd = [
                gt_val,
                af_val,
                dp_val
            ]
            
            # Estende a lista de informações das amostras dessa variante
            samples_fltrd_line.extend(sample_fltrd)
        
        # Une as amostras ao restante das informações
        fltrd_line.extend(samples_fltrd_line)
        data.append(fltrd_line)
    
    df = pd.DataFrame(data, columns=multi_cols)
    return df

def df_to_csv(df, file_name):
    output_folder = 'output'
    name = file_name.replace('.vcf', '').replace('.gz', '') + '.csv'
    folder_path = os.path.join(output_folder, name)
    os.makedirs(output_folder, exist_ok=True)
    df.to_csv(folder_path, index=False)
    

# df_raw = vcf_to_df_raw(FILE_PATH, VCF_FILE)
# df_raw

# df_filtered_INFO = vcf_to_df_filtered_INFO(VCF_FILE)
# df_filtered_INFO.head()

df_filtered_Samples = vcf_to_df_filtered_Samples(VCF_FILE)
df_to_csv(df_filtered_Samples, FILE_NAME)
df_filtered_Samples.iloc[0:20, 0:16]

# Sugestão de melhoria: para arquivos muitos grandes, vale a pena fazer o processamento com chunks

In [ ]:
import sqlite3

table_name = FILE_NAME.replace('.csv', '') 

# Achatamento para o caso de colunas multiindex
df_raw.columns = [
    '_'.join(col).strip('_') if isinstance(col, tuple) else col 
    for col in df_raw.columns.values
]

# Conectar e salvar no banco
with sqlite3.connect('genomic.db') as connection:
    print(f"Successfully connected to database!")
    
    #apend ou replace é a melhor opção?
    df_raw.to_sql(table_name, connection, if_exists='append', index=False)
    
    print(f"Data from {FILE_NAME} successfully saved on '{table_name}'!")